# UC Credential Passthrough — Read & Write Demo

**Purpose:** Demonstrates the full read/write lifecycle of the UC Passthrough library — from session initialisation through format-specific reads, transformations, and writes back to ADLS or Unity Catalog tables.

**Key principle:** The library analyses each *path* at call-time and routes automatically:
- Paths inside UC volumes or Delta tables → **Unity Catalog** (governance enforced, no credential injection needed)
- Raw `abfss://` paths outside UC objects → **ADLS direct** with the caller's user token
- `uc_passthrough_override` option → explicit escape hatch in either direction

---
**Files covered in this notebook:**
| File | Role |
|---|---|
| `uc_passthrough_library.py` | Session wrapper — intercepts `spark.read` and exposes `patch_dataframe_write()` |
| `uc_passthrough_writer.py` | Write-side proxy — mirrors `DataFrameWriter` with full routing |
| `direct_adls_writer.py` | ADLS writer — CSV / JSON / text / parquet / ORC / Avro via PyArrow |
| `direct_adls_reader.py` | ADLS reader — same formats, same PyArrow approach |
| `authentication_manager.py` | Token acquisition & ADLS client factory |
| `path_analyzer.py` | Deterministic UC vs ADLS routing logic |

---
## 0 · Environment setup

Replace the placeholder values with your Azure AD app registration details.  
The library files must be on `sys.path` — in Databricks this is easiest via a Workspace file path or by attaching the library to the cluster.

In [ ]:
import sys
import os

# ── point to wherever you've placed the library files ───────────────────────
LIBRARY_PATH = "/Workspace/Shared/uc_passthrough"   # adjust for your workspace
if LIBRARY_PATH not in sys.path:
    sys.path.insert(0, LIBRARY_PATH)

# ── ADLS paths used throughout this notebook ────────────────────────────────
STORAGE_ACCOUNT = "<your_storage_account>"           # e.g. dlsgskprod
CONTAINER       = "<your_container>"                 # e.g. raw
BASE_PATH       = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

DEMO_INPUT_PATH  = f"{BASE_PATH}/passthrough_demo/input"
DEMO_OUTPUT_PATH = f"{BASE_PATH}/passthrough_demo/output"

# ── UC table references ──────────────────────────────────────────────────────
UC_CATALOG  = "<your_catalog>"                       # e.g. gsk_prod
UC_SCHEMA   = "<your_schema>"                        # e.g. clinical_data
UC_TABLE    = f"{UC_CATALOG}.{UC_SCHEMA}.passthrough_demo_output"

print("Environment variables set.")
print(f"  Input path  : {DEMO_INPUT_PATH}")
print(f"  Output path : {DEMO_OUTPUT_PATH}")
print(f"  UC table    : {UC_TABLE}")

---
## 1 · Initialise authentication and wrap the Spark session

`UCPassthroughDataFrameReader` wraps the native `SparkSession`.  After wrapping:
- `spark.read` → intercepted and routed through the library
- `spark.patch_dataframe_write(df)` → wraps a DataFrame so `df.write` is also intercepted
- Everything else (`spark.sql`, `spark.catalog`, `spark.createDataFrame`, …) passes straight through to the underlying session unchanged.

In [ ]:
from authentication_manager import AuthenticationManager
from path_analyzer import PathAnalyzer
from uc_passthrough_library import UCPassthroughDataFrameReader

# ── authentication ───────────────────────────────────────────────────────────
auth_config = {
    "client_id"     : dbutils.secrets.get(scope="kv-gsk", key="sp-client-id"),
    "client_secret" : dbutils.secrets.get(scope="kv-gsk", key="sp-client-secret"),
    "tenant_id"     : dbutils.secrets.get(scope="kv-gsk", key="tenant-id"),
    # Set True to use the Spark session's on-behalf-of token (interactive clusters)
    "use_interactive_flow": True,
    "cache_tokens"  : True,
}

auth_manager = AuthenticationManager(auth_config)
ok = auth_manager.initialize_user_context(spark)
print(f"Authenticated as: {auth_manager.get_current_user_upn()}  (ok={ok})")

# ── path routing config ──────────────────────────────────────────────────────
# force_uc_patterns  : regex list — paths that must always go through UC
# force_adls_patterns: regex list — paths that must always go direct
routing_config = {
    "force_uc_patterns"  : [r"/Volumes/"],       # all UC Volume paths → UC
    "force_adls_patterns": [r"passthrough_demo"], # demo paths → ADLS direct
}
path_analyzer = PathAnalyzer(config=routing_config)

# ── wrap the session ─────────────────────────────────────────────────────────
spark = UCPassthroughDataFrameReader(spark, auth_manager, path_analyzer)
print(f"Session wrapped: {repr(spark)}")

---
## 2 · Seed demo data

Write a small synthetic dataset to ADLS directly via the Pandas SDK (no Spark credentials needed here) so the rest of the notebook has something to read.  
In a real workflow this data already exists — skip this cell.

In [ ]:
import pandas as pd
import io
from urllib.parse import urlparse

# Sample pharmaceutical trial data — three subjects, two visits each
SEED_ROWS = [
    {"subject_id": "SUBJ-001", "visit": "V1", "compound": "GSK123", "dose_mg": 100, "response_score": 3.2, "site": "UK"},
    {"subject_id": "SUBJ-001", "visit": "V2", "compound": "GSK123", "dose_mg": 100, "response_score": 4.1, "site": "UK"},
    {"subject_id": "SUBJ-002", "visit": "V1", "compound": "GSK123", "dose_mg": 200, "response_score": 5.0, "site": "US"},
    {"subject_id": "SUBJ-002", "visit": "V2", "compound": "GSK123", "dose_mg": 200, "response_score": 6.3, "site": "US"},
    {"subject_id": "SUBJ-003", "visit": "V1", "compound": "GSK456", "dose_mg": 150, "response_score": 2.8, "site": "IN"},
    {"subject_id": "SUBJ-003", "visit": "V2", "compound": "GSK456", "dose_mg": 150, "response_score": 3.5, "site": "IN"},
]
seed_df = pd.DataFrame(SEED_ROWS)

# Upload as CSV to the input path
parsed   = urlparse(DEMO_INPUT_PATH)
container, account_host = parsed.netloc.split("@")
fs_client = auth_manager.get_adls_client(
    f"https://{account_host}"
).get_file_system_client(container)

blob_path = parsed.path.lstrip("/") + "/trial_data.csv"
buf = io.StringIO()
seed_df.to_csv(buf, index=False)
file_client = fs_client.get_file_client(blob_path)
file_client.upload_data(buf.getvalue().encode(), overwrite=True)

print(f"Seeded {len(seed_df)} rows → {DEMO_INPUT_PATH}/trial_data.csv")
seed_df

---
## 3 · Read — CSV (ADLS direct path)

The path analyser sees `passthrough_demo` in the path and classifies it as ADLS direct (forced by the `force_adls_patterns` config above).  
The library fetches the data using the caller's user token — no service principal credential injection into Spark.

In [ ]:
# Identical syntax to native spark.read — the proxy is transparent
df_trials = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load(f"{DEMO_INPUT_PATH}/trial_data.csv")
)

print(f"Schema: {df_trials.dtypes}")
df_trials.show()

In [ ]:
# ── Shorthand form — identical behaviour ─────────────────────────────────────
df_trials_v2 = spark.read.csv(
    f"{DEMO_INPUT_PATH}/trial_data.csv",
    header=True,
    inferSchema=True,
)
df_trials_v2.show()

---
## 4 · Read — multi-path load (union across files)

Passing a **list of paths** to `.load()` returns a single unioned DataFrame — each path is routed independently so paths on different access methods can coexist in the same call.

In [ ]:
# Imagine two monthly extract files sitting in ADLS
paths = [
    f"{DEMO_INPUT_PATH}/trial_data.csv",   # exists — seeded above
    # f"{DEMO_INPUT_PATH}/trial_data_jan.csv",  # uncomment when you have more files
]

df_multi = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load(paths)
)
print(f"Rows loaded across {len(paths)} path(s): {df_multi.count()}")
df_multi.show()

---
## 5 · Read — UC table (Unity Catalog governed)

`.table()` is always routed through Unity Catalog regardless of any config overrides — governance is non-negotiable for named tables.

In [ ]:
# Replace with a real UC table in your catalog
REFERENCE_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.compound_metadata"

df_compounds = spark.read.table(REFERENCE_TABLE)
df_compounds.show()

---
## 6 · Transform — join ADLS-sourced and UC-governed data

The resulting DataFrame has mixed lineage — columns from an ADLS direct read joined with columns from a UC table.  
The write router ignores this entirely: **routing is determined solely by the output path** you hand to `.save()`.

In [ ]:
from pyspark.sql.functions import col, round as spark_round, avg, count

# Join trial results with compound metadata from UC
df_enriched = (
    df_trials
    .join(df_compounds, on="compound", how="left")
    .select(
        "subject_id",
        "visit",
        "compound",
        "site",
        col("dose_mg").cast("integer"),
        spark_round(col("response_score"), 2).alias("response_score"),
        "compound_class",   # from UC table
        "mechanism",        # from UC table
    )
)
df_enriched.show()

In [ ]:
# Aggregate — mean response and visit count per compound + site
df_summary = (
    df_enriched
    .groupBy("compound", "site", "compound_class")
    .agg(
        avg("response_score").alias("mean_response"),
        count("*").alias("n_observations"),
    )
    .orderBy("compound", "site")
)
df_summary.show()

---
## 7 · Write — Parquet to ADLS direct path

`spark.patch_dataframe_write(df)` wraps any DataFrame so its `.write` property returns the passthrough proxy.  
The output path is classified as ADLS direct by the path analyser, so PyArrow serialises the data and the library uploads it with the user's token.

In [ ]:
PARQUET_OUTPUT = f"{DEMO_OUTPUT_PATH}/enriched_trials.parquet"

# Wrap the DataFrame so .write is passthrough-aware
df_enriched_w = spark.patch_dataframe_write(df_enriched)

(
    df_enriched_w.write
                 .mode("overwrite")
                 .parquet(PARQUET_OUTPUT)
)
print(f"Written: {PARQUET_OUTPUT}")

---
## 8 · Read back what was written — round-trip verification

Reading the parquet we just wrote confirms the routing is symmetric: the same path analyser classifies the input and output paths identically.

In [ ]:
df_rt = spark.read.parquet(PARQUET_OUTPUT)
print(f"Round-trip row count: {df_rt.count()} (expected {df_enriched.count()})")
df_rt.show()

# Schema must match
assert df_rt.schema == df_enriched.schema, "Schema mismatch after round-trip!"
print("Schema round-trip: OK")

---
## 9 · Write — partitioned Parquet (Hive-style layout)

`.partitionBy()` works identically to native Spark.  The ADLS writer creates the `col=value/part-00000.parquet` directory structure expected by Spark, Hive, and Delta's `MSCK REPAIR TABLE`.

In [ ]:
PARTITIONED_OUTPUT = f"{DEMO_OUTPUT_PATH}/enriched_by_site"

df_enriched_w2 = spark.patch_dataframe_write(df_enriched)

(
    df_enriched_w2.write
                  .mode("overwrite")
                  .partitionBy("site")
                  .parquet(PARTITIONED_OUTPUT)
)
print(f"Partitioned output written: {PARTITIONED_OUTPUT}")

# Read back one partition directly
df_uk = spark.read.parquet(f"{PARTITIONED_OUTPUT}/site=UK")
print("UK partition:")
df_uk.show()

---
## 10 · Write — CSV with full option forwarding

Write options (`sep`, `header`, `nullValue`, `compression`) are stripped of any sensitive auth keys and forwarded to the underlying writer unchanged.

In [ ]:
CSV_OUTPUT = f"{DEMO_OUTPUT_PATH}/summary.csv"

df_summary_w = spark.patch_dataframe_write(df_summary)

(
    df_summary_w.write
                .mode("overwrite")
                .option("header", True)
                .option("sep", ",")
                .option("nullValue", "")
                .csv(CSV_OUTPUT)
)
print(f"CSV written: {CSV_OUTPUT}")

# Read back
df_csv_rt = spark.read.csv(CSV_OUTPUT, header=True, inferSchema=True)
df_csv_rt.show()

---
## 11 · Write — JSON Lines (ADLS direct)

The JSON writer defaults to JSON Lines format (one object per row) — the same convention Spark's native JSON writer uses.

In [ ]:
JSON_OUTPUT = f"{DEMO_OUTPUT_PATH}/summary.json"

df_summary_w2 = spark.patch_dataframe_write(df_summary)

(
    df_summary_w2.write
                 .mode("overwrite")
                 .json(JSON_OUTPUT)
)
print(f"JSON written: {JSON_OUTPUT}")

# Read back
df_json_rt = spark.read.json(JSON_OUTPUT)
df_json_rt.show()

---
## 12 · Write — saveAsTable (Unity Catalog governed)

`saveAsTable` always routes through UC — the path analyser is never consulted.  
This is the right way to persist a derived DataFrame as a permanent, governed Delta table.

In [ ]:
df_summary_w3 = spark.patch_dataframe_write(df_summary)

(
    df_summary_w3.write
                 .mode("overwrite")
                 .format("delta")
                 .saveAsTable(UC_TABLE)
)
print(f"Table written: {UC_TABLE}")

# Verify via UC — table() always reads through UC governance
df_uc_verify = spark.read.table(UC_TABLE)
print(f"UC table row count: {df_uc_verify.count()}")
df_uc_verify.show()

---
## 13 · Write — insertInto an existing UC table

`insertInto` appends rows to an existing table.  Like `saveAsTable`, it always routes through UC.  
Use `overwrite=True` to replace the table contents.

In [ ]:
# Simulate new data arriving — one extra subject
new_rows = spark.createDataFrame([
    ("GSK123", "US", "Kinase inhibitor", 4.9, 3),
], schema=["compound", "site", "compound_class", "mean_response", "n_observations"])

new_rows_w = spark.patch_dataframe_write(new_rows)
new_rows_w.write.insertInto(UC_TABLE, overwrite=False)

print(f"After insert — row count: {spark.read.table(UC_TABLE).count()}")

---
## 14 · Write — explicit routing override

The `uc_passthrough_override` option bypasses path analysis and forces a specific route.  
Useful when a path would normally be classified as ADLS direct but you need UC governance for a specific operation.

In [ ]:
# Force UC routing for a raw ADLS path (unusual — shown for completeness)
df_forced_uc = spark.patch_dataframe_write(df_summary)

(
    df_forced_uc.write
                .mode("overwrite")
                .option("uc_passthrough_override", "uc")   # force UC
                .parquet(f"{DEMO_OUTPUT_PATH}/forced_uc_write")
)
print("Forced UC write complete.")

# ── Force ADLS direct on the read side ───────────────────────────────────────
# Useful when a UC volume path should bypass UC for performance
df_forced_adls = (
    spark.read
         .format("parquet")
         .option("uc_passthrough_override", "adls")    # force ADLS direct
         .load(f"{DEMO_OUTPUT_PATH}/enriched_trials.parquet")
)
print(f"Forced ADLS direct read: {df_forced_adls.count()} rows")

---
## 15 · Write mode behaviour

All four Spark write modes are enforced via `WriteTransactionContext` — writes go to a temp path first and are committed (or rolled back) atomically.

In [ ]:
from pyspark.sql import Row

TEST_PATH = f"{DEMO_OUTPUT_PATH}/mode_test.parquet"

df_small = spark.createDataFrame([Row(x=1), Row(x=2)])
df_small_w = spark.patch_dataframe_write(df_small)

# ── error (default) — raises if target exists ─────────────────────────────
df_small_w.write.mode("error").parquet(TEST_PATH)    # first write: succeeds
print("mode=error  first write: OK")

try:
    df_small_w.write.mode("error").parquet(TEST_PATH) # second write: should raise
except Exception as e:
    print(f"mode=error  second write raised as expected: {type(e).__name__}")

# ── ignore — silently skips if target exists ──────────────────────────────
df_small_w.write.mode("ignore").parquet(TEST_PATH)
print("mode=ignore : target exists, write skipped silently")

# ── append — adds a new part file alongside existing ─────────────────────
df_small_w.write.mode("append").parquet(TEST_PATH)
print(f"mode=append : row count now {spark.read.parquet(TEST_PATH).count()} (was 2)")

# ── overwrite — replaces target ───────────────────────────────────────────
df_overwrite = spark.createDataFrame([Row(x=99)])
df_overwrite_w = spark.patch_dataframe_write(df_overwrite)
df_overwrite_w.write.mode("overwrite").parquet(TEST_PATH)
result = spark.read.parquet(TEST_PATH)
assert result.count() == 1 and result.first().x == 99
print(f"mode=overwrite: row count = {result.count()}, value = {result.first().x}")

---
## 16 · Passthrough transparency — native Spark APIs still work

The session wrapper never breaks standard Spark operations.  `spark.sql`, `spark.catalog`, `spark.createDataFrame`, and `spark.udf` all pass straight through.

In [ ]:
# spark.sql — passes straight through
df_sql = spark.sql(f"SELECT * FROM {UC_TABLE} WHERE mean_response > 4.0")
df_sql.show()

# spark.catalog — passes straight through
tables = spark.catalog.listTables(f"{UC_CATALOG}.{UC_SCHEMA}")
print(f"Tables in {UC_CATALOG}.{UC_SCHEMA}: {[t.name for t in tables[:5]]}")

# spark.createDataFrame — passes straight through
df_inline = spark.createDataFrame([("A", 1), ("B", 2)], ["name", "value"])
df_inline.show()

---
## 17 · Diagnostic — path routing introspection

Call `path_analyzer.analyze_path()` directly to see exactly how a path will be routed before committing a read or write.  Useful during development and when debugging unexpected routing behaviour.

In [ ]:
test_paths = [
    (f"{DEMO_INPUT_PATH}/trial_data.csv",              "csv"),
    (f"{DEMO_OUTPUT_PATH}/enriched_trials.parquet",    "parquet"),
    (f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/files/a.csv", "csv"),
    (f"{UC_CATALOG}.{UC_SCHEMA}.some_table",           "table"),
]

print(f"{'PATH':<65} {'FORMAT':<10} {'ROUTE':<8} REASONING")
print("-" * 120)

for path, fmt in test_paths:
    method, analysis = path_analyzer.analyze_path(path=path, format_type=fmt)
    reasoning = "; ".join(analysis["reasoning"])
    short_path = (path[:62] + "...") if len(path) > 65 else path
    print(f"{short_path:<65} {fmt:<10} {method.upper():<8} {reasoning}")

---
## 18 · Cleanup (optional)

Remove the demo output files and the UC table created in this notebook.

In [ ]:
# ── Remove ADLS demo files ────────────────────────────────────────────────────
from urllib.parse import urlparse

parsed   = urlparse(DEMO_OUTPUT_PATH)
container, account_host = parsed.netloc.split("@")
fs_client = auth_manager.get_adls_client(
    f"https://{account_host}"
).get_file_system_client(container)

output_blob = parsed.path.lstrip("/")
for p in fs_client.get_paths(path=output_blob, recursive=True):
    if not p.is_directory:
        fs_client.get_file_client(p.name).delete_file()
        print(f"  deleted: {p.name}")

print("ADLS demo files cleaned up.")

# ── Drop the UC table ─────────────────────────────────────────────────────────
spark.sql(f"DROP TABLE IF EXISTS {UC_TABLE}")
print(f"UC table {UC_TABLE} dropped.")